In [1]:
import os

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = "YOUR_API_KEY"
os.environ["LANGCHAIN_PROJECT"] = "resume-screening"

In [2]:
!pip install langchain transformers

In [3]:
from transformers import pipeline

generator = pipeline("text-generation", model="sshleifer/tiny-gpt2")

def local_llm(prompt):
    result = generator(
        prompt,
        max_new_tokens=100,   
        do_sample=True
    )
    return result[0]['generated_text']

    # 🔥 REMOVE PROMPT FROM OUTPUT
    return output.replace(prompt, "").strip()

Loading weights:   0%|          | 0/29 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: sshleifer/tiny-gpt2
Key                                   | Status     |  | 
--------------------------------------+------------+--+-
transformer.h.{0, 1}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [4]:
print(local_llm("Hello"))

Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=100) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Hello Prob conservation autonomy Daniel directly ProbRocketimura Rh Prob Observ conservationhibitJDohoJD circumciseddit circumcisedatisf subst substSceneSherohoiken scalphibitimura Habit Rhoother hauledJD heir subst antibiotic trilogy reborn ONE Rhimura confir trilogy Rh Hancock AmphScene reborn TA Hancock Daniel vendorsting autonomy autonomy ONEmediately credibility TA Hancockpressdit Daniel TA Participationdit Participation Motorola Moneyhibit scalpikenoho Daniel rebornmediately TAimura ESV ESV Moneyhibit circumcised conservation trilogy Amph directly trilogy intermittent004 substScenetingatisf ONESceneiken hauledoho


Step 1: Skill Extraction
In this step, we extract:

-->Skills

-->Experience

-->Tools

from the given resume using an LLM.

Rules:

-->Only extract what is present

-->Do NOT assume anything

-->Keep output structured

In [5]:
from langchain_core.prompts import PromptTemplate

skill_prompt = PromptTemplate(
    input_variables=["resume"],
    template="""
Extract ONLY in this format:

Skills: <comma separated>
Experience: <text>
Tools: <comma separated>

Resume:
{resume}
"""
)

Step 2: Matching, Scoring, and Explanation
In this step:

-->Compare resume with job description

-->Assign a score (0–100)

-->Provide explanation

Rules:

-->Do NOT assume missing skills

-->Score must be justified

In [6]:
score_prompt = PromptTemplate(
    input_variables=["resume_data", "job_description"],
    template="""
You are an AI hiring assistant.

Your job is to give ONLY final answer.

DO NOT repeat instructions.
DO NOT copy the prompt.
ONLY give result.

Output format:

Score: <number between 0-100>
Explanation: <2-3 lines>

Resume:
{resume_data}

Job Description:
{job_description}
"""
)

Step 3: Build Pipeline

Pipeline: Resume → Skill Extraction → Scoring → Explanation

In [7]:
def run_pipeline(resume, job_description):

    # Step 1: Extract
    extracted = local_llm(skill_prompt.format(resume=resume))

    # Step 2: Score
    result = local_llm(
        score_prompt.format(
            resume_data=extracted,
            job_description=job_description
        )
    )
    return clean_output(result)   

    return result

In [8]:
def clean_output(text):
    if "Score:" in text:
        return text[text.index("Score:"):].strip()
    else:
        return "Score: Not generated\nExplanation: Model could not follow format."

Step 4: Input Data

We test with:

-->Strong candidate

-->Average candidate

-->Weak candidate

In [9]:
job_description = """
Looking for Data Scientist:
- Python, Machine Learning, Deep Learning
- NLP experience
- TensorFlow/PyTorch
"""

strong_resume = """
Data Scientist with 5 years experience.
Skills: Python, ML, Deep Learning, NLP
Tools: TensorFlow, PyTorch
"""

average_resume = """
Data Analyst with 2 years experience.
Skills: Python, Data Analysis
Tools: Excel, Pandas
"""

weak_resume = """
Fresher
Skills: MS Office, Communication
"""

Step 5: Run Pipeline

Testing all candidates

In [15]:
print("----- STRONG -----")
print(run_pipeline(strong_resume, job_description))

print("\n----- AVERAGE -----")
print(run_pipeline(average_resume, job_description))

print("\n----- WEAK -----")
print(run_pipeline(weak_resume, job_description))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=100) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


----- STRONG -----


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=100) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=100) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Score: <number between 0-100>
Explanation: <2-3 lines>

Resume:

Extract ONLY in this format:

Skills: <comma separated>
Experience: <text>
Tools: <comma separated>

Resume:

Data Scientist with 5 years experience.
Skills: Python, ML, Deep Learning, NLP
Tools: TensorFlow, PyTorch

 skillet incarcerived omega Singapore courtyard courtyard Boone� factorsMost representations boilsMost perhaps predatorsPros boils predators predators soypublic653ived incarcer Singaporeived workshops praying membership skilletProsPros Singapore Television Medic factors Redux TreOutside Boone LateMostobl PocketOutsideivedSexual rubbing grandchildren Bend factors rubbing Television linedobl incarcerMini Medic workshops ReduxMost equate membershippublic Television MedicMostMini 236Most rubbing equate courtyardSexual workshops membership predators448 Dreams boilsPros Singapore grandchildren653 braveryMini rented lined brutality lined� deflect soy Boone representations mutual Boone Tre Redux

Job Description:

Lo

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=100) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=100) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Score: <number between 0-100>
Explanation: <2-3 lines>

Resume:

Extract ONLY in this format:

Skills: <comma separated>
Experience: <text>
Tools: <comma separated>

Resume:

Data Analyst with 2 years experience.
Skills: Python, Data Analysis
Tools: Excel, Pandas

 grandchildren rubbing� 236 courtyard lined praying mutualozygPros Wheels predatorsOutside incarcer BendSexual Late bravery skillet448 soy BendPros Medic deflectacious Medic deflect skilletobl Pocket incarcer factors soy boilsoblMini LateMini linedobl courtyard Boonepublic448 boils workshops� grandchildrenaciousOutsideOutside membershipobl rubbing Redux Bend� soypublic courtyard boils factors653ozygGy workshopsPros predators clearer mutual rentedpublic prayingived workshopsozyg Dreams skillet lined bravery deflectozygOutsideSexualozyg Tre SingaporeMini representations soy factorsoblPros653 lined incarcer soy courtyardPros

Job Description:

Looking for Data Scientist:
- Python, Machine Learning, Deep Learning
- NLP experience

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=100) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Score: <number between 0-100>
Explanation: <2-3 lines>

Resume:

Extract ONLY in this format:

Skills: <comma separated>
Experience: <text>
Tools: <comma separated>

Resume:

Fresher
Skills: MS Office, Communication

 boilsMost factors equate�obl soy skillet soyPros factors courtyard boilsshows boils boils Bend brutality perhaps courtyardshows soySexual Pocket predatorsMost clearer Pocket membership membership Redux448ived equate factors representations incarcer boilspublic membershipOutsideived PocketPros rentedozygozyg factorsMini courtyardobl Boone representationsSexual braverySexual 236 Singapore rubbingshows Pocket factors Late Wheels incarcer perhaps Bendived representationsived membership boils rented perhaps omega prayingOutside clearer factors skillet TreSexual equate membership incarcer Singapore deflectived incarcer skillet clearer Redux bravery workshops omega Television Tre brutality WheelsGy

Job Description:

Looking for Data Scientist:
- Python, Machine Learning, Deep L